# 01 — Data Audit du dataset Tetouan

Projet : Smart City Energy Forecasting — Tetouan  
Objectif : charger le dataset brut, vérifier sa structure, contrôler la qualité des données et préparer les premières variables cibles.

## Importation des bibliothèques

In [5]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.max_columns", None)

## Définition du chemin et du renommage des colonnes

In [6]:
DATA_PATH = Path('../data/raw/Tetuan City power consumption.csv')

COLUMN_MAPPING = {
    "DateTime": "datetime",
    "Temperature": "temperature",
    "Humidity": "humidity",
    "Wind Speed": "wind_speed",
    "general diffuse flows": "general_diffuse_flows",
    "diffuse flows": "diffuse_flows",
    "Zone 1 Power Consumption": "zone1_power",
    "Zone 2  Power Consumption": "zone2_power",
    "Zone 3  Power Consumption": "zone3_power"
}

## Fonction de chargement et de préparation initiale


In [7]:
def load_prepare_data(filepath, column_mapping):
    """
    Charge le dataset Tetouan, renomme les colonnes, vérifie la qualité
    de la date, configure l'index temporel et prépare les variables cibles.
    """
    df = pd.read_csv(filepath)

    print(f"Dimensions brutes : {df.shape}")
    print("Colonnes originales :")
    print(df.columns.tolist())

    # Renommage des colonnes
    df = df.rename(columns=column_mapping)

    expected_columns = {
        "datetime",
        "temperature",
        "humidity",
        "wind_speed",
        "general_diffuse_flows",
        "diffuse_flows",
        "zone1_power",
        "zone2_power",
        "zone3_power"
    }

    missing_columns = expected_columns - set(df.columns)
    if missing_columns:
        raise ValueError(f"Colonnes manquantes après renommage : {missing_columns}")

    # Conversion correcte de la date
    df["datetime"] = pd.to_datetime(
        df["datetime"],
        format="%m/%d/%Y %H:%M",
        errors="coerce"
    )

    invalid_dates = df["datetime"].isna().sum()
    if invalid_dates > 0:
        raise ValueError(f"{invalid_dates} dates n'ont pas pu être converties.")

    # Contrôle des doublons avant suppression
    n_duplicates = df.duplicated(subset="datetime").sum()
    print(f"Doublons temporels détectés : {n_duplicates}")

    # Tri et suppression éventuelle des doublons
    df = df.sort_values("datetime").reset_index(drop=True)
    df = df.drop_duplicates(subset="datetime", keep="first")

    # Index temporel
    df = df.set_index("datetime")

    # Variables cibles
    df["target"] = df["zone1_power"]
    df["total_load"] = df["zone1_power"] + df["zone2_power"] + df["zone3_power"]

    return df

data = load_prepare_data(DATA_PATH, COLUMN_MAPPING)
print(f"Dimensions du dataset : {data.shape}")
display(data.head())


Dimensions brutes : (52416, 9)
Colonnes originales :
['DateTime', 'Temperature', 'Humidity', 'Wind Speed', 'general diffuse flows', 'diffuse flows', 'Zone 1 Power Consumption', 'Zone 2  Power Consumption', 'Zone 3  Power Consumption']
Doublons temporels détectés : 0
Dimensions du dataset : (52416, 10)


,temperature,humidity,wind_speed,general_diffuse_flows,diffuse_flows,zone1_power,zone2_power,zone3_power,target,total_load
datetime,,,,,,,,,,
2017-01-01 00:00:00,6.559,73.8,0.083,0.051,0.119,34055.69620,16128.87538,20240.96386,34055.69620,70425.53544
2017-01-01 00:10:00,6.414,74.5,0.083,0.070,0.085,29814.68354,19375.07599,20131.08434,29814.68354,69320.84387
2017-01-01 00:20:00,6.313,74.5,0.080,0.062,0.100,29128.10127,19006.68693,19668.43373,29128.10127,67803.22193
2017-01-01 00:30:00,6.121,75.0,0.083,0.091,0.096,28228.86076,18361.09422,18899.27711,28228.86076,65489.23209
2017-01-01 00:40:00,5.921,75.7,0.081,0.048,0.085,27335.69620,17872.34043,18442.40964,27335.69620,63650.44627


## Fonction d'audit de la qualité des données

In [8]:
def audit_data_quality(df):
    """
    Génère un audit de qualité du dataset :
    dimensions, période, fréquence, valeurs manquantes,
    doublons temporels et statistiques descriptives.
    """
    print("=" * 60)
    print("AUDIT GLOBAL DU DATASET")
    print("=" * 60)

    print(f"Nombre d'observations : {df.shape[0]}")
    print(f"Nombre de colonnes : {df.shape[1]}")

    print(f"Début de période : {df.index.min()}")
    print(f"Fin de période : {df.index.max()}")

    inferred_freq = pd.infer_freq(df.index)
    print(f"Fréquence temporelle inférée : {inferred_freq}")

    total_missing = df.isna().sum().sum()
    print(f"Nombre total de valeurs manquantes : {total_missing}")

    duplicated_index = df.index.duplicated().sum()
    print(f"Doublons temporels dans l'index : {duplicated_index}")

    audit = pd.DataFrame({
        "Type": df.dtypes.astype(str),
        "Valeurs Manquantes": df.isna().sum(),
        "Taux de Manquants (%)": (df.isna().mean() * 100).round(2)
    })

    summary = df.describe().T
    audit = audit.join(summary[["min", "mean", "max"]], how="left")

    return audit

# Exécution de l'audit
audit_results = audit_data_quality(data)
display(audit_results)

AUDIT GLOBAL DU DATASET
Nombre d'observations : 52416
Nombre de colonnes : 10
Début de période : 2017-01-01 00:00:00
Fin de période : 2017-12-30 23:50:00
Fréquence temporelle inférée : 10min
Nombre total de valeurs manquantes : 0
Doublons temporels dans l'index : 0


,Type,Valeurs Manquantes,Taux de Manquants (%),min,mean,max
temperature,float64,0,0.0,3.247000,18.810024,40.01000
humidity,float64,0,0.0,11.340000,68.259518,94.80000
wind_speed,float64,0,0.0,0.050000,1.959489,6.48300
general_diffuse_flows,float64,0,0.0,0.004000,182.696614,1163.00000
diffuse_flows,float64,0,0.0,0.011000,75.028022,936.00000
zone1_power,float64,0,0.0,13895.696200,32344.970564,52204.39512
zone2_power,float64,0,0.0,8560.081466,21042.509082,37408.86076
zone3_power,float64,0,0.0,5935.174070,17835.406218,47598.32636
target,float64,0,0.0,13895.696200,32344.970564,52204.39512
total_load,float64,0,0.0,36785.039739,71222.885864,134208.14595
